# Statistical Arbitrage: NIFTY Futures vs NIFTYBEES
## Cointegration and Error-Correction — complete calculation notebook

Every number in the presentation is produced by this notebook. Run **Kernel → Restart & Run All**.

**Inputs** (same folder): `NIFTY1_15min.csv`, `NIFTYBEES_15min.csv` — 15-minute bars exported from TradingView (NSE feed).
**Outputs**: `results.json` (every headline figure) and `daily_panel.csv` (the data used by the Excel replication workbook).

| Step | Research step | Method |
|---|---|---|
| 1 | Stationarity testing | ADF, KPSS |
| 2 | Hedge-ratio estimation | OLS (Engle–Granger stage 1) |
| 3 | Cointegration testing | Engle–Granger residual test, Johansen trace test |
| 4 | Specification diagnosis | Deterministic-trend regression, cost-of-carry decomposition |
| 5 | Model re-specification | Trend-augmented cointegrating regression |
| 6 | Error-correction modelling | ECM with HAC standard errors, half-life |
| 7 | Signal construction and walk-forward backtest | Rolling z-score, cost model, capital accounting |

In [1]:
import json, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 20)
R = {}                       # every headline number is stored here -> results.json
TD = 252                     # trading-day annualisation convention used throughout

### 0 · Load and clean the data
Drop indicator columns, drop the 15:30 post-close stamps (bars are stamped by their **open** time, so 15:15 is the last valid bar),
inner-join the two legs on timestamp, drop Muhurat evening bars, and take the **last 15-minute close of each session**.

In [2]:
def load(path, tag):
    df = pd.read_csv(path, usecols=["time", "open", "high", "low", "close", "Volume"])
    df["dt"] = pd.to_datetime(df["time"], unit="s", utc=True).dt.tz_convert("Asia/Kolkata")
    return df.rename(columns={"close": tag})[["dt", tag]]

fut, bees = load("NIFTY1_15min.csv", "F"), load("NIFTYBEES_15min.csv", "B")
log = {"raw_bars_F": len(fut), "raw_bars_B": len(bees)}
fut  = fut[fut.dt.dt.strftime("%H:%M") != "15:30"]
bees = bees[bees.dt.dt.strftime("%H:%M") != "15:30"]
m = fut.merge(bees, on="dt", how="inner")
m = m[m.dt.dt.strftime("%H:%M").between("09:15", "15:15")].sort_values("dt")
log["aligned_bars"] = len(m)
d = (m.set_index("dt").resample("1D").last().dropna().reset_index())
d["date"] = d.dt.dt.tz_localize(None).dt.normalize()
d = d[["date", "F", "B"]].reset_index(drop=True)
log["daily_obs"] = len(d)
print(pd.Series(log).to_string())
print("sample:", d.date.iloc[0].date(), "->", d.date.iloc[-1].date())
R["n"] = len(d); R["start"] = str(d.date.iloc[0].date()); R["end"] = str(d.date.iloc[-1].date())
R["aligned_bars"] = log["aligned_bars"]

raw_bars_F      20233
raw_bars_B      20231
aligned_bars    20220
daily_obs         811
sample: 2023-05-02 -> 2026-08-07


In [3]:
d["lnF"], d["lnB"] = np.log(d.F), np.log(d.B)
d["dlnF"], d["dlnB"] = d.lnF.diff(), d.lnB.diff()
d["t"] = np.arange(len(d))
rho = d[["dlnF", "dlnB"]].corr().iloc[0, 1]
R["corr_raw"] = round(rho, 4); R["ratio_mean"] = round((d.F / d.B).mean(), 4)
print(f"Correlation of daily log returns : {rho:.4f}")
print(f"Mean price ratio F/B             : {(d.F/d.B).mean():.4f}   (contract-size assumption 65:6500 implies 100.0)")

Correlation of daily log returns : 0.9459
Mean price ratio F/B             : 89.6795   (contract-size assumption 65:6500 implies 100.0)


### Step 1 · Stationarity testing — ADF and KPSS
**ADF regression:** $\Delta y_t = \alpha + \gamma\,y_{t-1} + \sum_{i=1}^{k}\delta_i\,\Delta y_{t-i} + \varepsilon_t$ ,
$H_0:\gamma=0$ (unit root) vs $H_1:\gamma<0$. Lag length $k$ chosen by AIC (max lag $=12(n/100)^{1/4}$).
**KPSS** reverses the null: $H_0$ = stationary. Both are reported so the conclusion does not rest on one test's null.

In [4]:
def adf_row(x, name, reg="c"):
    s, p, k, nobs, cv, _ = adfuller(x.dropna(), regression=reg, autolag="AIC")
    ks, kp, _, kcv = kpss(x.dropna(), regression="c", nlags="auto")
    return dict(series=name, ADF=round(s, 3), p=round(p, 4), lags=k, nobs=nobs,
                cv5=round(cv["5%"], 3), KPSS=round(ks, 3), KPSS_cv5=kcv["5%"])
t1 = pd.DataFrame([adf_row(d.lnF, "ln F  (level)"), adf_row(d.lnB, "ln B  (level)"),
                   adf_row(d.dlnF, "Δ ln F (1st diff)"), adf_row(d.dlnB, "Δ ln B (1st diff)")])
print(t1.to_string(index=False))
R["step1"] = t1.to_dict("records")
for nm, x in [("lnF", d.lnF), ("lnB", d.lnB)]:
    s, p, *_ = adfuller(x, regression="ct", autolag="AIC")
    R[f"adf_ct_{nm}"] = [round(s, 3), round(p, 4)]
print("\nWith constant + trend:", {k: v for k, v in R.items() if k.startswith("adf_ct")})

           series     ADF      p  lags  nobs    cv5  KPSS  KPSS_cv5
    ln F  (level)  -2.309 0.1690     5   805 -2.865 3.059     0.463
    ln B  (level)  -2.367 0.1512     0   810 -2.865 3.258     0.463
Δ ln F (1st diff) -12.865 0.0000     4   805 -2.865 0.220     0.463
Δ ln B (1st diff) -13.347 0.0000     3   806 -2.865 0.271     0.463

With constant + trend: {'adf_ct_lnF': [-2.051, 0.5735], 'adf_ct_lnB': [-1.937, 0.6354]}


### Step 2 · Hedge-ratio estimation — OLS (Engle–Granger stage 1)
$\ln F_t = \alpha + \beta\,\ln B_t + \varepsilon_t$. With I(1) regressors OLS is *super-consistent* for $\beta$, but its $t$-statistics are
non-standard — which is why Step 3 must use Engle–Granger (MacKinnon) critical values, not ordinary ADF ones.

In [5]:
eg1 = sm.OLS(d.lnF, sm.add_constant(d.lnB)).fit()
eg1r = sm.OLS(d.lnB, sm.add_constant(d.lnF)).fit()
print(eg1.summary().tables[1])
print(f"R² = {eg1.rsquared:.5f}   n = {int(eg1.nobs)}   reverse-direction slope = {eg1r.params.iloc[1]:.5f}")
R["alpha"], R["beta"] = round(eg1.params.iloc[0], 5), round(eg1.params.iloc[1], 5)
R["beta_se"] = round(eg1.bse.iloc[1], 5); R["r2"] = round(eg1.rsquared, 5); R["beta_rev"] = round(eg1r.params.iloc[1], 5)

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.9618      0.012    419.025      0.000       4.939       4.985
lnB            0.9161      0.002    429.599      0.000       0.912       0.920
R² = 0.99564   n = 811   reverse-direction slope = 1.08679


### Step 3a · Cointegration — Engle–Granger residual test
$\Delta\hat\varepsilon_t = \rho\,\hat\varepsilon_{t-1} + \sum\phi_i\,\Delta\hat\varepsilon_{t-i} + u_t$ ,  $H_0:\rho=0$ (no cointegration).
Critical values are MacKinnon's for an **estimated** cointegrating vector (2 variables).

In [6]:
def eg_row(y, x, name, trend="c"):
    s, p, cv = coint(y, x, trend=trend, autolag="aic")
    return dict(direction=name, trend=trend, EG=round(s, 3), p=round(p, 4),
                cv1=round(cv[0], 3), cv5=round(cv[1], 3), cv10=round(cv[2], 3))
t3 = pd.DataFrame([eg_row(d.lnF, d.lnB, "ln F on ln B"), eg_row(d.lnB, d.lnF, "ln B on ln F")])
print(t3.to_string(index=False))
# the residual ADF that coint() runs internally (needed for the Excel replication)
res_c = eg1.resid
s, p, k, *_ = adfuller(res_c, regression="n", autolag="aic")
print(f"\nResidual ADF (no constant) = {s:.3f}, lags = {k}  -> identical statistic to coint()")
R["step3"] = t3.to_dict("records"); R["eg_resid_lags"] = int(k)

   direction trend     EG      p   cv1    cv5  cv10
ln F on ln B     c -3.096 0.0891 -3.91 -3.344 -3.05
ln B on ln F     c -3.037 0.1018 -3.91 -3.344 -3.05

Residual ADF (no constant) = -3.096, lags = 3  -> identical statistic to coint()


### Step 3b · Johansen trace test
$\lambda_{trace}(r) = -T\sum_{i=r+1}^{n}\ln(1-\hat\lambda_i)$ ;  $H_0$: at most $r$ cointegrating vectors. VAR in differences with 1 lag.

In [7]:
rows = []
for det, lab in [(0, "constant"), (1, "constant + trend")]:
    j = coint_johansen(np.column_stack([d.lnF, d.lnB]), det_order=det, k_ar_diff=1)
    for r_ in range(2):
        rows.append(dict(spec=lab, H0=f"r <= {r_}", trace=round(j.lr1[r_], 3), cv5=round(j.cvt[r_, 1], 3),
                         reject=bool(j.lr1[r_] > j.cvt[r_, 1]), eigen=round(j.eig[r_], 5)))
tj = pd.DataFrame(rows); print(tj.to_string(index=False)); R["johansen"] = rows

            spec     H0   trace    cv5  reject   eigen
        constant r <= 0  28.496 15.494    True 0.02732
        constant r <= 1   6.086  3.842    True 0.00749
constant + trend r <= 0 134.001 18.398    True 0.14853
constant + trend r <= 1   3.918  3.842    True 0.00483

### Step 4 · Specification diagnosis
If a stable relationship exists, $\ln(F/B)$ should fluctuate around a constant. Regress it on time: $\ln(F_t/B_t) = a + b\,t + e_t$.

In [8]:
d["lnR"] = d.lnF - d.lnB
dg = sm.OLS(d.lnR, sm.add_constant(d.t)).fit()
R["diag_slope_ann"] = round(dg.params.iloc[1] * TD * 100, 3); R["diag_t"] = round(dg.tvalues.iloc[1], 1)
R["diag_r2"] = round(dg.rsquared, 4)
R["ratio_first"], R["ratio_last"] = round(d.F.iloc[0]/d.B.iloc[0], 2), round(d.F.iloc[-1]/d.B.iloc[-1], 2)
print(f"slope = {dg.params.iloc[1]:.3e} per trading day  = {R['diag_slope_ann']:.2f}% per year")
print(f"t = {R['diag_t']}   R² = {R['diag_r2']}   ratio {R['ratio_first']} -> {R['ratio_last']}")

slope = -4.609e-05 per trading day  = -1.16% per year
t = -106.5   R² = 0.9334   ratio 91.84 -> 88.15


### Step 4b · Cost-of-carry decomposition
Theory: $F_t = S_t\,e^{(r-q)\tau_t}$ and the ETF accumulates dividends, $B_t \propto S_t\,e^{q t}$, so
$\ln(F_t/B_t) = c + (r-q)\,\tau_t - q\,t$. Regress on days-to-expiry (DTE, calendar days) and $t$ (trading days):
annualised basis $r-q = 365\,a$, dividend drift $q = -252\,b$, implied financing rate $r = (r-q) + q$.

Monthly expiry: last **Thursday** up to Aug-2025, last **Tuesday** from Sep-2025 (NSE); a holiday moves expiry to the **previous** session.

In [9]:
sess = pd.Series(d.date)
sset = set(sess)
cut = pd.Timestamp("2025-09-01")
expiries = []
for (yy, mo), _ in sess.groupby([sess.dt.year, sess.dt.month]):
    me = pd.Timestamp(year=yy, month=mo, day=1) + pd.offsets.MonthEnd(0)
    wd = 3 if me < cut else 1
    e = me
    while e.dayofweek != wd:
        e -= pd.Timedelta(days=1)
    if e <= sess.iloc[-1]:
        k = 0
        while e not in sset and k < 7:
            e -= pd.Timedelta(days=1); k += 1
    expiries.append(e)
expiries = sorted(set(expiries))
exp_arr = np.array(expiries, dtype="datetime64[ns]")
nxt = np.searchsorted(exp_arr, d.date.values)                       # next expiry on or after each date
d["expiry"] = exp_arr[np.clip(nxt, 0, len(exp_arr) - 1)]
d["DTE"] = (d.expiry - d.date).dt.days
d["is_expiry"] = d.date.isin(expiries)
d["splice"] = d.is_expiry.shift(1, fill_value=False)                # first session after expiry = contract switch
cc = sm.OLS(d.lnR, sm.add_constant(d[["DTE", "t"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
basis = cc.params["DTE"] * 365; q = -cc.params["t"] * TD; r_impl = basis + q
jump = (d.dlnF - d.dlnB)[d.splice]
print(cc.summary().tables[1])
print(f"annualised basis (r-q) = {basis*100:.2f}%   dividend drift q = {q*100:.2f}%   implied financing rate r = {r_impl*100:.2f}%")
print(f"R² = {cc.rsquared:.4f} | expiries in sample: {int(d.is_expiry.sum())} | contract-switch jumps: mean {jump.mean()*100:.3f}%, positive {int((jump>0).sum())}/{len(jump)}")
R.update(basis_ann=round(basis*100, 2), q_ann=round(q*100, 2), r_impl=round(r_impl*100, 2),
         carry_r2=round(cc.rsquared, 4), carry_t_dte=round(cc.tvalues["DTE"], 1), carry_t_t=round(cc.tvalues["t"], 1),
         n_expiries=int(d.is_expiry.sum()), splice_mean=round(jump.mean()*100, 3), splice_pos=int((jump > 0).sum()))

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.5127      0.000   9839.970      0.000       4.512       4.514
DTE            0.0001   1.47e-05     10.004      0.000       0.000       0.000
t          -4.618e-05   8.27e-07    -55.815      0.000   -4.78e-05   -4.46e-05
annualised basis (r-q) = 5.37%   dividend drift q = 1.16%   implied financing rate r = 6.54%
R² = 0.9476 | expiries in sample: 39 | contract-switch jumps: mean 0.579%, positive 39/39


### Step 5 · Model re-specification — trend-augmented cointegrating regression
$\ln F_t = \alpha + \beta\,\ln B_t + \delta\,t + \varepsilon_t$. **Stated before estimation:** theory requires $\beta \to 1$ and $\delta \to -q$.

In [10]:
Xt = sm.add_constant(pd.DataFrame({"lnB": d.lnB, "t": d.t}))
eg5 = sm.OLS(d.lnF, Xt).fit()
s5, p5, cv5 = coint(d.lnF, d.lnB, trend="ct", autolag="aic")
s5r, p5r, _ = coint(d.lnB, d.lnF, trend="ct", autolag="aic")
_, _, k5, *_ = adfuller(eg5.resid, regression="n", autolag="aic")
print(eg5.summary().tables[1])
print(f"R² = {eg5.rsquared:.5f} | trend δ = {eg5.params['t']:.3e}/day = {eg5.params['t']*TD*100:.2f}% per year")
print(f"EG (constant+trend) = {s5:.3f}, p = {p5:.2e}, 5% cv = {cv5[1]:.3f}, residual-ADF lags = {k5} | reverse direction EG = {s5r:.3f}")
R.update(beta5=round(eg5.params["lnB"], 5), beta5_se=round(eg5.bse["lnB"], 5), alpha5=round(eg5.params["const"], 5),
         delta5=float(f"{eg5.params['t']:.4e}"), delta5_ann=round(eg5.params["t"]*TD*100, 2), r2_5=round(eg5.rsquared, 5),
         eg5=round(s5, 3), eg5_p=float(f"{p5:.2e}"), eg5_cv5=round(cv5[1], 3), eg5_cv1=round(cv5[0], 3),
         eg5_rev=round(s5r, 3), eg5_lags=int(k5))
d["resid5"], d["resid_naive"] = eg5.resid.values, eg1.resid.values

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.5312      0.009    500.169      0.000       4.513       4.549
lnB            0.9970      0.002    594.073      0.000       0.994       1.000
t          -4.493e-05   7.73e-07    -58.091      0.000   -4.64e-05   -4.34e-05
R² = 0.99916 | trend δ = -4.493e-05/day = -1.13% per year
EG (constant+trend) = -8.060, p = 7.99e-11, 5% cv = -3.792, residual-ADF lags = 3 | reverse direction EG = -8.014


### Step 6 · Error-correction model
$\Delta\ln F_t = c + \gamma\,\hat\varepsilon_{t-1} + u_t$ with Newey–West (HAC, 10 lags) standard errors.
Error correction requires $\gamma<0$ and significant. Half-life (Ornstein–Uhlenbeck approximation) $= \ln 2 / |\gamma|$;
the exact discrete-time value $\ln 0.5/\ln(1+\gamma)$ is reported alongside.

In [11]:
e = pd.DataFrame({"dy": d.dlnF, "ect": d.resid5.shift(1)}).dropna()
ecm = sm.OLS(e.dy, sm.add_constant(e.ect)).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
ecm_ols = sm.OLS(e.dy, sm.add_constant(e.ect)).fit()
g = ecm.params["ect"]
print(ecm.summary().tables[1])
print(f"γ = {g:.4f} | HAC t = {ecm.tvalues['ect']:.3f}, p = {ecm.pvalues['ect']:.4f} | OLS t = {ecm_ols.tvalues['ect']:.3f}")
print(f"half-life: ln2/|γ| = {np.log(2)/-g:.2f} days  |  exact discrete = {np.log(.5)/np.log(1+g):.2f} days")
R.update(gamma=round(g, 4), gamma_t=round(ecm.tvalues["ect"], 3), gamma_p=round(ecm.pvalues["ect"], 4),
         gamma_t_ols=round(ecm_ols.tvalues["ect"], 3), halflife=round(np.log(2)/-g, 2),
         halflife_discrete=round(np.log(.5)/np.log(1+g), 2), ecm_c=float(f"{ecm.params['const']:.4e}"))

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0004      0.000      1.238      0.216      -0.000       0.001
ect           -0.4832      0.146     -3.302      0.001      -0.770      -0.196
γ = -0.4832 | HAC t = -3.302, p = 0.0010 | OLS t = -4.741
half-life: ln2/|γ| = 1.43 days  |  exact discrete = 1.05 days


### Step 7a · Inputs for the backtest
* **Roll-adjusted futures return.** On the first session after each expiry the continuous contract switches to the next month
  and jumps by the new contract's basis — a splice, not a price move. On those days the futures return is set to the ETF return
  less one day of carry. Signals still use raw prices.
* **Risk-free rate:** RBI policy repo path. **Costs:** statutory components (STT, stamp duty) plus exchange/regulatory charges and slippage.

In [12]:
d["dlnF_adj"] = np.where(d.splice, d.dlnB - r_impl / TD, d.dlnF)
repo = [("2023-02-08", .0650), ("2025-02-07", .0625), ("2025-04-09", .0600), ("2025-06-06", .0550), ("2025-12-05", .0525)]
d["rf"] = np.nan
for dt_, v in repo:
    d.loc[d.date >= dt_, "rf"] = v
d["stt_F"] = np.select([d.date <= "2024-09-30", d.date <= "2026-03-31"], [1.25e-4, 2.0e-4], 5.0e-4)
rho_adj = np.corrcoef(d.dlnF_adj.iloc[1:], d.dlnB.iloc[1:])[0, 1]; R["corr_adj"] = round(rho_adj, 4)
print(f"correlation of daily log returns: raw {R['corr_raw']:.4f} | contract switches removed {rho_adj:.4f}")
print("time-weighted average repo over the sample:", round(d.rf.mean()*100, 3), "%")
print(d.groupby(d.stt_F)["date"].agg(["min", "max", "count"]))
R["rf_avg"] = round(d.rf.mean()*100, 3)
COST = {  # per side, as a fraction of notional
    "retail":        dict(stamp_F=2e-5, oth_F=3.5e-5, slip_F=1.0e-4, stamp_B=1.5e-4, stt_B=1e-5, oth_B=5e-5, slip_B=1.5e-4),
    "institutional": dict(stamp_F=2e-5, oth_F=2.0e-5, slip_F=0.5e-4, stamp_B=1.5e-4, stt_B=1e-5, oth_B=3e-5, slip_B=0.75e-4)}

correlation of daily log returns: raw 0.9459 | contract switches removed 0.9559
time-weighted average repo over the sample: 6.054 %
                min        max  count
stt_F                                
0.000125 2023-05-02 2024-09-30    353
0.000200 2024-10-01 2026-03-30    370
0.000500 2026-04-01 2026-08-07     88


### Step 7b · Walk-forward signal
Each day $t$ the cointegrating regression $\ln F = \alpha+\beta\ln B+\delta t$ is re-estimated on the **previous 250 sessions only**
(minimum 60) and applied to day $t$ out-of-sample: $z_t = \hat\varepsilon_t/\hat\sigma_t$. No information from $t$ or later enters.

In [13]:
def rolling_z(df, window=250, min_obs=60, trend=True):
    y, x, tt = df.lnF.values, df.lnB.values, df.t.values.astype(float)
    n = len(df); out = np.full((n, 3), np.nan)
    for i in range(1, n):
        lo = max(0, i - window)
        if i - lo < min_obs: continue
        cols = [np.ones(i - lo), x[lo:i]] + ([tt[lo:i]] if trend else [])
        X = np.column_stack(cols); b, *_ = np.linalg.lstsq(X, y[lo:i], rcond=None)
        res = y[lo:i] - X @ b; sig = np.sqrt((res**2).sum() / (len(res) - X.shape[1]))
        fit = b[0] + b[1]*x[i] + (b[2]*tt[i] if trend else 0)
        out[i] = [b[1], (y[i] - fit) / sig, sig]
    return pd.DataFrame(out, columns=["beta_r", "z", "sig"], index=df.index)

def positions(z, entry=2.0, exit_=0.5, stop=3.5, max_hold=10, allow="both"):
    pos = np.zeros(len(z)); cur = 0.0; held = 0; ev = [""] * len(z)
    for i, zi in enumerate(z):
        if not np.isfinite(zi): cur, held = 0.0, 0; continue
        if cur != 0:
            held += 1
            if abs(zi) > stop or held >= max_hold or abs(zi) < exit_:
                cur, held, ev[i] = 0.0, 0, "exit"
        if cur == 0 and entry < abs(zi) <= stop:
            side = -np.sign(zi)
            if allow == "both" or (allow == "short" and side < 0) or (allow == "long" and side > 0):
                cur, held, ev[i] = side, 0, "entry"
        pos[i] = cur
    return pos, ev

sig = rolling_z(d)
d = d.join(sig)
R["pct_z2"] = round(100 * (d.z.abs() > 2).mean(), 1); R["pct_z3"] = round(100 * (d.z.abs() > 3).mean(), 1)
print(f"|z| > 2 on {R['pct_z2']}% of days, |z| > 3 on {R['pct_z3']}%   (first signal day: {d.date[d.z.notna()].iloc[0].date()})")

|z| > 2 on 6.7% of days, |z| > 3 on 1.0%   (first signal day: 2023-07-26)


### Step 7c · Backtest — the signal's own quality (both directions, retail costs)
Daily excess P&L per ₹1 of futures notional (position decided at close $t-1$):

$x_t = p_{t-1}\big[\Delta\ln F^{adj}_t - \beta_h\,\Delta\ln B_t + \beta_h\,r^f_t/252\big] - c_t$

The $r^f$ term makes the spread **carry-neutral**: the ETF leg is financed (or its short proceeds invested) at the risk-free rate,
so $x_t$ is the return *above* the risk-free rate. $\beta_h$ is the hedge ratio fixed at entry.

In [14]:
def backtest(df, pos, costs, fund_rate=None):
    c = COST[costs]; n = len(df)
    bh = np.zeros(n)
    for i in range(n):
        if pos[i] != 0:
            bh[i] = bh[i-1] if (i > 0 and pos[i-1] == pos[i]) else df.beta_r.values[i]
    qF, qB = pos, -bh * pos
    dF, dB = np.diff(np.r_[0, qF]), np.diff(np.r_[0, qB])
    stt = df.stt_F.values
    buyF, sellF = np.maximum(dF, 0), np.maximum(-dF, 0)
    buyB, sellB = np.maximum(dB, 0), np.maximum(-dB, 0)
    cost = (buyF*(c["stamp_F"]+c["oth_F"]+c["slip_F"]) + sellF*(stt+c["oth_F"]+c["slip_F"])
            + buyB*(c["stamp_B"]+c["oth_B"]+c["slip_B"]) + sellB*(c["stt_B"]+c["oth_B"]+c["slip_B"]))
    roll = (df.is_expiry.values & (pos != 0)) * ((c["stamp_F"]+c["oth_F"]+c["slip_F"]) + (stt+c["oth_F"]+c["slip_F"]))
    cost = cost + roll
    p1, b1 = np.r_[0, pos[:-1]], np.r_[0, bh[:-1]]
    rate = df.rf.values if fund_rate is None else fund_rate
    spread = p1 * (df.dlnF_adj.fillna(0).values - b1 * df.dlnB.fillna(0).values)
    x = spread + p1 * b1 * rate / TD - cost
    return pd.DataFrame({"pos": pos, "bh": bh, "spread": spread, "cost": cost, "x": x, "p1": p1, "b1": b1}, index=df.index)

def stats(x, pos, ev):
    eq = np.cumsum(x); dd = eq - np.maximum.accumulate(np.r_[0, eq])[1:]
    trades = int(sum(1 for e_ in ev if e_ == "entry")); live = np.r_[0, pos[:-1]] != 0
    return dict(trades=trades, ann=round(float(x.mean())*TD*100, 3), vol=round(float(x.std(ddof=1))*np.sqrt(TD)*100, 3),
                sharpe=round(float(x.mean()/x.std(ddof=1)*np.sqrt(TD)), 2), mdd=round(float(dd.min())*100, 3),
                in_mkt=round(100*float(np.mean(pos != 0)), 1), hit=round(100*float(np.mean(x[live] > 0)), 1),
                total=round(float(eq[-1])*100, 3), t_sharpe=round(float(x.mean()/x.std(ddof=1)*np.sqrt(TD)*np.sqrt(len(x)/TD)), 2))

pos_both, ev_both = positions(d.z.values)
bt = backtest(d, pos_both, "retail")
gross = bt.spread + bt.p1 * bt.b1 * d.rf / TD
S = stats(bt.x.values, pos_both, ev_both); Sg = stats(gross.values, pos_both, ev_both)
print(pd.DataFrame({"gross (before costs)": Sg, "net (after costs)": S}).T.to_string())
print(f"average cost per round trip: {bt.cost.sum()/max(S['trades'],1)*1e4:.1f} bp of notional")
R["signal_net"], R["signal_gross"] = S, Sg
R["cost_rt_bp"] = round(bt.cost.sum()/max(S["trades"], 1)*1e4, 1)
d["eq_net"], d["eq_gross"] = np.cumsum(bt.x.values), np.cumsum(gross.values)

                      trades    ann    vol  sharpe    mdd  in_mkt   hit  total  t_sharpe
gross (before costs)    29.0  1.603  1.322    1.21 -0.687    21.1  51.5  5.160      2.18
net (after costs)       29.0  0.515  1.328    0.39 -1.280    21.1  51.5  1.656      0.70


average cost per round trip: 12.1 bp of notional


### Step 7d · Four people, one signal — capital accounting
Notional $N$ = one lot = 65 × final futures price. Futures margin $m$ = 13% of notional (SPAN + exposure margin, assumption).
Everyone's idle cash earns the repo rate, so each result is reported as return **above** the risk-free rate.

| | Trades allowed | Capital (per ₹1 of notional) | Extra cost |
|---|---|---|---|
| Aarav | short-spread only (retail cannot hold an ETF short overnight) | ETF 1.00 + margin 0.13 | — |
| Meera | as Aarav | own 0.25 of ETF (MTF) + margin 0.13 | MTF interest on 0.75 of ETF while a trade is open |
| Rohan | short futures against the ETF he already holds | 50% cash leg of margin = 0.065 | — (no ETF trades) |
| Kavya | both directions, institutional costs | margin after 70% cross-margin relief = 0.039 | funding at repo + 0.75% |

In [15]:
LOT, M_, MTF, PRIME = 65, 0.13, 0.14, 0.0075
N = float(d.F.iloc[-1] * LOT); yrs = len(d) / TD
pos_s, ev_s = positions(d.z.values, allow="short")
btA = backtest(d, pos_s, "retail")
live_s = np.r_[0, pos_s[:-1]] != 0
xA = btA.x.values
xM = xA - live_s * 0.75 * btA.b1.values * (MTF - d.rf.values) / TD
# Rohan: only the futures leg trades; increment vs buy-and-hold = short futures P&L - futures costs
cR = COST["retail"]; dFR = np.diff(np.r_[0, pos_s])
stt = d.stt_F.values
costR = (np.maximum(dFR, 0)*(cR["stamp_F"]+cR["oth_F"]+cR["slip_F"]) + np.maximum(-dFR, 0)*(stt+cR["oth_F"]+cR["slip_F"])
         + (d.is_expiry.values & (pos_s != 0))*((cR["stamp_F"]+cR["oth_F"]+cR["slip_F"]) + (stt+cR["oth_F"]+cR["slip_F"])))
xR = np.r_[0, pos_s[:-1]] * d.dlnF_adj.fillna(0).values - costR
btK = backtest(d, pos_both, "institutional", fund_rate=(d.rf + PRIME).values)
xK = btK.x.values

def char(name, x, cap, pos, ev):
    st = stats(x, pos, ev); prof = x.sum() / yrs * N
    return dict(name=name, capital_per_N=cap, capital_rs=round(cap*N), profit_rs=round(prof),
                excess_on_capital=round(100 * prof / (cap*N), 2), ann_on_notional=st["ann"], sharpe=st["sharpe"],
                mdd_on_capital=round(st["mdd"] / cap, 2), trades=st["trades"], in_mkt=st["in_mkt"])
C = [char("Aarav", xA, 1 + M_, pos_s, ev_s), char("Meera", xM, 0.25 + M_, pos_s, ev_s),
     char("Rohan", xR, 0.5 * M_, pos_s, ev_s), char("Kavya", xK, M_ * 0.30, pos_both, ev_both)]
tc = pd.DataFrame(C); print(f"N = ₹{N:,.0f} (65 × {d.F.iloc[-1]:,.1f}) | years = {yrs:.3f} | avg repo = {d.rf.mean()*100:.2f}%\n")
print(tc.to_string(index=False))
bhB = d.dlnB.fillna(0).values
R["N"] = round(N); R["years"] = round(yrs, 3); R["chars"] = C
R["rohan_bh"] = round(bhB.mean()*TD*100, 2); R["rohan_ov"] = round((bhB + xR).mean()*TD*100, 2)
R["rohan_bh_dd"] = round((lambda e: (e - np.maximum.accumulate(e)).min())(np.cumsum(bhB))*100, 2)
R["rohan_ov_dd"] = round((lambda e: (e - np.maximum.accumulate(e)).min())(np.cumsum(bhB + xR))*100, 2)
R["mtf_drag_ann"] = round((live_s * 0.75 * btA.b1.values * (MTF - d.rf.values)).mean()*100, 3)
print(f"\nRohan: buy-and-hold {R['rohan_bh']}%/yr  ->  with overlay {R['rohan_ov']}%/yr (log-return basis)")

N = ₹1,602,620 (65 × 24,655.7) | years = 3.218 | avg repo = 6.05%

 name  capital_per_N  capital_rs  profit_rs  excess_on_capital  ann_on_notional  sharpe  mdd_on_capital  trades  in_mkt
Aarav          1.130     1810961       3430               0.19            0.214    0.22           -1.08      15    13.7
Meera          0.380      608996     -10035              -1.65           -0.626   -0.63           -6.04      15    13.7
Rohan          0.065      104170     -47906             -45.99           -2.989   -0.56         -249.23      15    13.7
Kavya          0.039       62502      12873              20.60            0.803    0.61          -29.26      29    21.1

Rohan: buy-and-hold 10.7%/yr  ->  with overlay 7.71%/yr (log-return basis)


### Stress tests

In [16]:
ST = {}
naive = rolling_z(d, trend=False); pn, en = positions(naive.z.values)
dn = d.drop(columns=["beta_r", "z", "sig"]).join(naive); ST["naive"] = stats(backtest(dn, pn, "retail").x.values, pn, en)
x_ex = bt.x.values.copy(); x_ex[(d.date == "2024-06-04").values] = 0; ST["ex_election"] = stats(x_ex, pos_both, ev_both)
h = len(d)//2; ST["half1"] = stats(bt.x.values[:h], pos_both[:h], ev_both[:h]); ST["half2"] = stats(bt.x.values[h:], pos_both[h:], ev_both[h:])
d["pos"] = pos_both; ST["held_on_election"] = bool(d.loc[d.date == "2024-06-04", "pos"].iloc[0] != 0)
grid = {}
for en_ in (1.5, 2.0, 2.5):
    for ex_ in (0.0, 0.5, 1.0):
        pg, eg_ = positions(d.z.values, entry=en_, exit_=ex_); grid[f"{en_}/{ex_}"] = stats(backtest(d, pg, "retail").x.values, pg, eg_)["sharpe"]
ST["grid"] = grid; ST["grid_best"] = max(grid, key=grid.get)
xc2 = bt.spread + bt.p1*bt.b1*d.rf/TD - 2*bt.cost; ST["cost_x2"] = stats(xc2.values, pos_both, ev_both)
p1_, b1_ = np.r_[0, pos_both[:-1]], np.r_[0, bt.bh.values[:-1]]
raw_ = p1_ * (d.dlnF.fillna(0).values - b1_ * d.dlnB.fillna(0).values)
spl_ = d.splice.values & (p1_ != 0)
ST["splice"] = dict(gross_unadjusted=round(float(raw_.mean())*TD*100, 2), gross_adjusted=Sg["ann"], days=int(spl_.sum()),
                    booked=round(float(raw_[spl_].sum())*100, 2), long_on_splice=int((p1_[spl_] > 0).sum()))
pl_, el_ = positions(d.z.values, allow="long"); ST["rohan_switch"] = stats(backtest(d, pl_, "retail").x.values, pl_, el_)
tab = pd.DataFrame({k_: v_ for k_, v_ in ST.items() if isinstance(v_, dict) and "sharpe" in v_}).T[["trades", "ann", "sharpe", "mdd", "t_sharpe"]]
print(tab.to_string()); print("\nposition held on 4-Jun-2024 (election result):", ST["held_on_election"])
print("parameter grid (entry/exit → Sharpe):", ST["grid"], "| best:", ST["grid_best"])
print("contract-splice check:", ST["splice"])
R["stress"] = ST

              trades    ann  sharpe    mdd  t_sharpe
naive           33.0 -0.519   -0.31 -2.617     -0.55
ex_election     29.0  0.515    0.39 -1.280      0.70
half1           13.0 -0.014   -0.01 -1.280     -0.01
half2           16.0  1.042    0.73 -0.961      0.93
cost_x2         29.0 -0.574   -0.41 -1.990     -0.74
rohan_switch    16.0  0.396    0.41 -0.835      0.74

position held on 4-Jun-2024 (election result): False
parameter grid (entry/exit → Sharpe): {'1.5/0.0': -0.61, '1.5/0.5': 0.5, '1.5/1.0': 0.64, '2.0/0.0': -0.43, '2.0/0.5': 0.39, '2.0/1.0': 0.65, '2.5/0.0': -0.38, '2.5/0.5': 0.05, '2.5/1.0': 0.15} | best: 2.0/1.0
contract-splice check: {'gross_unadjusted': 3.74, 'gross_adjusted': 1.603, 'days': 9, 'booked': 5.79, 'long_on_splice': 9}


In [17]:
R["charts"] = {"date": d.date.dt.strftime("%Y-%m-%d").tolist(), "F100": (d.F/d.F.iloc[0]*100).round(3).tolist(),
               "B100": (d.B/d.B.iloc[0]*100).round(3).tolist(), "ratio": (d.F/d.B).round(4).tolist(),
               "res_naive": (d.resid_naive*100).round(4).tolist(), "res5": (d.resid5*100).round(4).tolist(),
               "z": d.z.round(3).tolist(), "eq_net": (d.eq_net*100).round(4).tolist(), "eq_gross": (d.eq_gross*100).round(4).tolist()}
json.dump(R, open("results.json", "w"), default=float, indent=1)
d[["date", "F", "B", "lnF", "lnB", "dlnF", "dlnB", "t", "DTE", "is_expiry", "splice", "dlnF_adj", "rf", "stt_F"]].to_csv("daily_panel.csv", index=False)
print("saved results.json and daily_panel.csv")

saved results.json and daily_panel.csv


### From β to contracts and units
β comes from a log–log regression, so it is a ratio of **rupee exposures**, not of units:
ETF notional = β × futures notional, and NIFTYBEES units = β × lot × F ÷ B.

In [18]:
def to_units(F, B, beta, lot=65):
    fut = lot * F; etf = beta * fut; u = round(etf / B)
    return dict(futures_notional=round(fut), etf_notional=round(etf), etf_units=u, rounding_err_pct=round(abs(u*B - etf)/fut*100, 4))
i0 = ev_both.index("entry")
print("First trade", d.date[i0].date(), "β at entry", round(bt.bh.values[i0], 4), to_units(d.F[i0], d.B[i0], bt.bh.values[i0]))
print("Today     ", d.date.iloc[-1].date(), "β (Step 5)", R["beta5"], to_units(d.F.iloc[-1], d.B.iloc[-1], R["beta5"]))
print("Naive 65 : 6,500 assumes F/B = 100; actual F/B today =", round(d.F.iloc[-1]/d.B.iloc[-1], 2))

First trade

 2023-07-26 β at entry 1.0559 {'futures_notional': 1285180, 'etf_notional': 1357021, 'etf_units': 6231, 'rounding_err_pct': np.float64(0.0071)}
Today      2026-08-07 β (Step 5) 0.99697 {'futures_notional': 1602620, 'etf_notional': 1597765, 'etf_units': 5712, 'rounding_err_pct': np.float64(0.0074)}
Naive 65 : 6,500 assumes F/B = 100; actual F/B today = 88.15


### Step 5b · Johansen with the trend restricted to the long-run relationship (closes Step 3's contradiction)
Step 5's model puts the trend **inside** the cointegrating relation: ln F − β ln B − δt is stationary. In Johansen's
framework that is **case 4** (restricted linear trend, unrestricted constant). statsmodels' `det_order=1` is a different
case (case 5: unrestricted trend, i.e. quadratic trends in levels), so case 4 is computed directly below.
Validation: the same routine in case 3 reproduces statsmodels `det_order=0` exactly.
5% critical values: MacKinnon, Haug & Michelis (1999): 25.87 (r = 0) and 12.52 (r ≤ 1).

In [19]:
def johansen_trace(Y, case, k=1):
    dY = np.diff(Y, axis=0); rows = range(k, len(Y) - 1)
    Z0 = np.array([dY[t] for t in rows]); lev = np.array([Y[t] for t in rows])
    lags = np.array([np.concatenate([dY[t - j] for j in range(1, k + 1)]) for t in rows])
    one = np.ones((len(Z0), 1)); tt = np.arange(len(Z0), dtype=float)[:, None]
    Z1, Z2 = (lev, np.hstack([one, lags])) if case == 3 else (np.hstack([lev, tt]), np.hstack([one, lags]))
    P = lambda A: A - Z2 @ np.linalg.lstsq(Z2, A, rcond=None)[0]
    R0, R1 = P(Z0), P(Z1); T = len(Z0)
    S00, S01, S11 = R0.T @ R0 / T, R0.T @ R1 / T, R1.T @ R1 / T
    lam = np.sort(np.real(np.linalg.eigvals(np.linalg.solve(S11, S01.T @ np.linalg.solve(S00, S01)))))[::-1][:Y.shape[1]]
    return [round(float(-T * np.log(1 - lam[r:]).sum()), 3) for r in range(Y.shape[1])], lam
Yj = d[["lnF", "lnB"]].values
print("validation, case 3:", johansen_trace(Yj, 3)[0], "vs statsmodels", coint_johansen(Yj, 0, 1).lr1.round(3).tolist())
tr4, lam4 = johansen_trace(Yj, 4); cv4 = [25.87, 12.52]
for r_, (t_, c_) in enumerate(zip(tr4, cv4)):
    print(f"H0: r <= {r_}   trace = {t_:8.3f}   5% critical = {c_}   ->  {'reject' if t_ > c_ else 'fail to reject'}")
print("Conclusion: exactly one cointegrating relationship (rank 1). The rank-2 contradiction of Step 3 disappears.")
R["johansen4"] = dict(trace=tr4, cv5=cv4, eig=[round(float(v), 5) for v in lam4])
json.dump(R, open("results.json", "w"), default=float, indent=1)

validation, case 3: [28.496, 6.086] vs statsmodels [28.496, 6.086]
H0: r <= 0   trace =  137.040   5% critical = 25.87   ->  reject
H0: r <= 1   trace =    6.611   5% critical = 12.52   ->  fail to reject
Conclusion: exactly one cointegrating relationship (rank 1). The rank-2 contradiction of Step 3 disappears.


## Step 7 at 15-minute resolution (the version used in the presentation)
The same walk-forward model (fitted on the previous 250 daily sessions only) is applied to **every 15-minute bar** of the next day.
A signal on one bar is traded at the **next** bar's close. **One trade at a time**; as soon as a trade closes, the next signal can be taken.
Futures leg = **synthetic future** (sell call + buy put at the ATM strike): STT is charged on the option premium, not the contract value.
Exit rules tested: exit when the gap closes (|z| < 0.5), or **hold to the monthly expiry**, when futures must converge to the index.

In [20]:
P15 = np.full((len(d), 4), np.nan); yv, xv, tv = d.lnF.values, d.lnB.values, d.t.values.astype(float)
for i in range(1, len(d)):
    lo = max(0, i - 250)
    if i - lo < 60: continue
    X_ = np.column_stack([np.ones(i - lo), xv[lo:i], tv[lo:i]]); b_, *_ = np.linalg.lstsq(X_, yv[lo:i], rcond=None)
    r_ = yv[lo:i] - X_ @ b_; P15[i] = [b_[0], b_[1], b_[2], np.sqrt((r_ ** 2).sum() / (len(r_) - 3))]
bars = m.copy(); bars["date"] = bars.dt.dt.tz_localize(None).dt.normalize()
bars["i"] = bars.date.map({v: k for k, v in enumerate(d.date)}); bars = bars.dropna(subset=["i"]).reset_index(drop=True); bars["i"] = bars.i.astype(int)
DI = bars.i.values; LF, LB, FP = np.log(bars.F.values), np.log(bars.B.values), bars.F.values; KB = len(bars)
pp_ = P15[DI]; Z = (LF - pp_[:, 0] - pp_[:, 1] * LB - pp_[:, 2] * DI) / pp_[:, 3]; bars["z"] = Z
ISEXP = d.is_expiry.values[DI]; ISLAST = np.r_[DI[1:] != DI[:-1], True]
dt_ = pd.to_datetime(d.date); STTO = np.select([dt_ <= "2024-09-30", dt_ <= "2026-03-31"], [6.25e-4, 1.0e-3], 1.5e-3)
STTX = np.where(dt_ <= "2026-03-31", 1.25e-3, 1.5e-3); PREM = 0.3989 * 0.14 * np.sqrt(np.maximum(d.DTE.values, 0) / 365)
RF = d.rf.values; CAL = np.r_[(d.date.values[1:] - d.date.values[:-1]).astype("timedelta64[D]").astype(int), 1]
ep = lambda zz: int(((np.abs(zz) > 2) & ~(np.abs(np.r_[0, zz[:-1]]) > 2)).sum())
print(f"15-minute bars: {KB:,} | signal episodes |z| > 2: daily checks {ep(np.nan_to_num(d.z.values))}, 15-minute checks {ep(np.nan_to_num(Z))}")
R["m15_bars"] = KB; R["m15_episodes"] = [ep(np.nan_to_num(d.z.values)), ep(np.nan_to_num(Z))]

15-minute bars: 20,220 | signal episodes |z| > 2: daily checks 39, 15-minute checks 228


In [21]:
def sim15(entry, exitx, allow, who, i_mtf=0.14, cost_mult=1.0, lag=1):
    c = COST["institutional" if who == "kavya" else "retail"]; bro = 1e-6 if who == "kavya" else 2.5e-6
    prime = 0.0075 if who == "kavya" else 0.0
    etf = lambda buy: (c["stamp_B"] + c["oth_B"] + c["slip_B"]) if buy else (c["stt_B"] + c["oth_B"] + c["slip_B"])
    daily = np.zeros(len(d)); trades = []; pos = 0; acc = 0.0; K = 0.0; tr = None
    for k in range(1, KB):
        i = DI[k]; v = 0.0
        if pos:
            v += pos * (LF[k] - LF[k-1]) if who == "rohan" else pos * ((LF[k] - LF[k-1]) - (LB[k] - LB[k-1]))
            zs = Z[k - lag]
            reason = "expiry" if (ISEXP[k] and ISLAST[k]) else ("closed" if (exitx is not None and np.isfinite(zs) and abs(zs) < exitx) else None)
            if reason:
                if reason == "expiry":
                    cst = STTX[i] * (max(K - FP[k], 0) if pos < 0 else max(FP[k] - K, 0)) / FP[k]
                else:
                    cst = STTO[i] * PREM[i] + 2 * bro + c["slip_F"]
                if who != "rohan": cst += etf(pos > 0)
                v -= cost_mult * cst; acc += v; daily[i] += v
                tr.update(k1=k, i1=i, Fx=FP[k], Bx=float(np.exp(LB[k])), reason=reason, net=acc); trades.append(tr); pos = 0
                continue
        zs = Z[k - lag]
        if not pos and not ISEXP[k] and not ISEXP[k-1] and np.isfinite(zs) and abs(zs) > entry:
            sgn = -int(np.sign(zs))
            if allow == "both" or sgn < 0:
                pos = sgn; K = FP[k]; acc = 0.0
                cst = STTO[i] * PREM[i] + 2 * bro + c["slip_F"] + 3e-5 * PREM[i]
                if who != "rohan": cst += etf(pos < 0)
                v -= cost_mult * cst
                tr = dict(who=who, side=pos, ks=k - lag, k0=k, i0=i, Fe=FP[k], Be=float(np.exp(LB[k])), zsig=float(zs))
        if pos and ISLAST[k]:
            if who in ("aarav", "meera"): v -= RF[i] * CAL[i] / 365
            if who == "meera": v -= 0.75 * (i_mtf - RF[i]) * CAL[i] / 365
            if who in ("kavya", "signal"): v += pos * (RF[i] + prime) * CAL[i] / 365
        daily[i] += v
        if pos: acc += v
    return daily, trades
CAP15 = dict(aarav=1 + 0.13, meera=0.25 + 0.13, rohan=0.5 * 0.13, kavya=0.13 * 0.30, signal=1.0)
ALLOW15 = dict(aarav="short", meera="short", rohan="short", kavya="both", signal="both")
H15 = len(d) // 2
def summ15(daily, trades, who):
    cap = CAP15[who] * R["N"]; prof = daily.sum() / (len(d) / TD) * R["N"]; sh = daily.mean() / daily.std(ddof=1) * np.sqrt(TD)
    eq = np.cumsum(daily); mdd = (eq - np.maximum.accumulate(np.r_[0, eq])[1:]).min(); tn = np.array([t_["net"] for t_ in trades])
    return dict(trades=len(trades), per_month=round(len(trades) / (len(d) / TD * 12), 1),
                avg_days=round(float(np.mean([(d.date[t_["i1"]] - d.date[t_["i0"]]).days for t_ in trades])), 1), wins=int((tn > 0).sum()),
                profit_rs=int(round(prof)), excess=round(100 * prof / cap, 2), h1=round(100 * daily[:H15].sum() / (H15 / TD) * R["N"] / cap, 2),
                h2=round(100 * daily[H15:].sum() / ((len(d) - H15) / TD) * R["N"] / cap, 2), sharpe=round(float(sh), 2),
                t=round(float(sh * np.sqrt(len(d) / TD)), 2), mdd_cap=round(100 * float(mdd) / CAP15[who], 1))

### Choosing each person's rule without hindsight
Six rules (entry 1.5σ / 2σ / 2.5σ × exit when closed / hold to expiry) are run for each person. The rule is **chosen on the first half
of the sample only** and judged on the second half, which it never saw.

In [22]:
RULES15 = [(e_, x_) for e_ in (1.5, 2.0, 2.5) for x_ in (0.5, None)]
lab15 = lambda r_: f"entry {r_[0]}σ · " + ("hold to expiry" if r_[1] is None else "exit when |z| < 0.5")
M15, TR15 = {}, {}
for who in ("aarav", "meera", "rohan", "kavya"):
    rows = []
    for r_ in RULES15:
        dly, trs = sim15(r_[0], r_[1], ALLOW15[who], who); s_ = summ15(dly, trs, who); s_["rule"] = lab15(r_); s_["_r"] = r_; rows.append(s_)
    T_ = pd.DataFrame(rows); best = T_.loc[T_.h1.idxmax()]
    dly, trs = sim15(best["_r"][0], best["_r"][1], ALLOW15[who], who); TR15[who] = trs
    M15[who] = {k_: (v_ if not isinstance(v_, (np.integer, np.floating)) else float(v_)) for k_, v_ in best.drop("_r").items()}
    M15[who]["entry"], M15[who]["exit"] = best["_r"]
    print(f"\n{who.upper()}  (capital ₹{CAP15[who] * R['N']:,.0f})"); print(T_[["rule", "trades", "per_month", "avg_days", "wins", "profit_rs", "excess", "h1", "h2", "sharpe"]].to_string(index=False))
    print(f"→ chosen on the first half: {best.rule} | unseen second half {best.h2}% | full sample {best.excess}%")
dly_s, TR15["signal"] = sim15(2.0, None, "both", "signal"); M15["signal"] = summ15(dly_s, TR15["signal"], "signal")
dly_g, trg = sim15(2.0, None, "both", "signal", cost_mult=0.0); M15["signal_gross"] = summ15(dly_g, trg, "signal")
print("\nSIGNAL (entry 2σ, hold to expiry, both directions, retail costs): net", M15["signal"]["excess"], "% | gross", M15["signal_gross"]["excess"], "% of notional a year")


AARAV  (capital ₹1,810,961)
                            rule  trades  per_month  avg_days  wins  profit_rs  excess    h1   h2  sharpe
entry 1.5σ · exit when |z| < 0.5      48        1.2       5.8    34      12872    0.71  0.13 1.29    0.65
     entry 1.5σ · hold to expiry      26        0.7      28.9    21      31185    1.72  1.34 2.10    0.55
entry 2.0σ · exit when |z| < 0.5      27        0.7       8.8    18      10949    0.60  0.07 1.14    0.66
     entry 2.0σ · hold to expiry      21        0.5      29.6    18      28224    1.56  1.17 1.94    0.80
entry 2.5σ · exit when |z| < 0.5      12        0.3      10.9     8       4863    0.27 -0.04 0.58    0.39
     entry 2.5σ · hold to expiry      11        0.3      28.7     9      16853    0.93  0.77 1.09    0.62
→ chosen on the first half: entry 1.5σ · hold to expiry | unseen second half 2.1% | full sample 1.72%

MEERA  (capital ₹608,996)


                            rule  trades  per_month  avg_days  wins  profit_rs  excess    h1    h2  sharpe
entry 1.5σ · exit when |z| < 0.5      48        1.2       5.8    25      -9821   -1.61 -3.67  0.44   -0.50
     entry 1.5σ · hold to expiry      26        0.7      28.9     5     -29708   -4.88 -5.72 -4.04   -0.52
entry 2.0σ · exit when |z| < 0.5      27        0.7       8.8    13      -8295   -1.36 -3.28  0.56   -0.50
     entry 2.0σ · hold to expiry      21        0.5      29.6     4     -22340   -3.67 -4.36 -2.98   -0.63
entry 2.5σ · exit when |z| < 0.5      12        0.3      10.9     6      -5842   -0.96 -2.15  0.22   -0.47
     entry 2.5σ · hold to expiry      11        0.3      28.7     3      -9189   -1.51 -2.16 -0.86   -0.34
→ chosen on the first half: entry 2.5σ · exit when |z| < 0.5 | unseen second half 0.22% | full sample -0.96%

ROHAN  (capital ₹104,170)
                            rule  trades  per_month  avg_days  wins  profit_rs  excess      h1     h2  sharpe
entry


KAVYA  (capital ₹62,502)
                            rule  trades  per_month  avg_days  wins  profit_rs  excess    h1    h2  sharpe
entry 1.5σ · exit when |z| < 0.5      72        1.9       5.5    50      35153   56.24 54.43 58.05    0.83
     entry 1.5σ · hold to expiry      34        0.9      25.1    25      27131   43.41 36.33 50.47    0.47
entry 2.0σ · exit when |z| < 0.5      44        1.1       7.0    27      22381   35.81 28.32 43.27    0.55
     entry 2.0σ · hold to expiry      29        0.8      24.8    21      32346   51.75 58.63 44.89    0.63
entry 2.5σ · exit when |z| < 0.5      19        0.5       7.7    12      16522   26.43 31.11 21.77    0.44
     entry 2.5σ · hold to expiry      15        0.4      25.7    12      30927   49.48 61.35 37.65    0.69
→ chosen on the first half: entry 2.5σ · hold to expiry | unseen second half 37.65% | full sample 49.48%

SIGNAL (entry 2σ, hold to expiry, both directions, retail costs): net 2.13 % | gross 2.76 % of notional a year


In [23]:
ROB15 = {}
for who in ("aarav", "meera", "rohan", "kavya"):
    e_, x_ = M15[who]["entry"], M15[who]["exit"]
    ROB15[who] = dict(same_bar=summ15(*sim15(e_, x_, ALLOW15[who], who, lag=0), who)["excess"],
                      costs_x2=summ15(*sim15(e_, x_, ALLOW15[who], who, cost_mult=2.0), who)["excess"])
ROB15["meera"]["mtf_10"] = summ15(*sim15(M15["meera"]["entry"], M15["meera"]["exit"], "short", "meera", i_mtf=0.10), "meera")["excess"]
dR, _ = sim15(M15["rohan"]["entry"], M15["rohan"]["exit"], "short", "rohan"); bh_ = d.dlnB.fillna(0).values
ROB15["rohan_port"] = [round(bh_.mean() * TD * 100, 2), round((bh_ + dR).mean() * TD * 100, 2)]
for k_, v_ in ROB15.items(): print(f"{k_:12s}", v_)
R["m15"] = dict(chars=M15, robust=ROB15)

aarav        {'same_bar': np.float64(1.93), 'costs_x2': np.float64(1.22)}
meera        {'same_bar': np.float64(0.63), 'costs_x2': np.float64(-1.74), 'mtf_10': np.float64(-0.08)}
rohan        {'same_bar': np.float64(51.2), 'costs_x2': np.float64(49.63)}
kavya        {'same_bar': np.float64(56.78), 'costs_x2': np.float64(43.78)}
rohan_port   [np.float64(10.7), np.float64(13.98)]


### Course-outline tests on the trades (sessions 3–12)
Descriptive shape, one-sample t-test and 95% CI, Wilcoxon signed-rank (non-parametric), proportion test on the win rate,
two-sample tests on the two halves (Welch t, Mann–Whitney, F-test for equal variances) and a chi-square test of association.

In [24]:
from statsmodels.stats.proportion import proportions_ztest
tn = pd.Series([t_["net"] for t_ in TR15["signal"]]) * 1e4; tside = pd.Series([t_["side"] for t_ in TR15["signal"]])
tdate = pd.Series([d.date[t_["i0"]] for t_ in TR15["signal"]]); half1 = tdate < d.date[H15]
tt_ = stats_ = None
from scipy import stats as sps
t1 = sps.ttest_1samp(tn, 0); ci_ = sps.t.interval(0.95, len(tn) - 1, tn.mean(), sps.sem(tn)); wil = sps.wilcoxon(tn)
pz = proportions_ztest(int((tn > 0).sum()), len(tn), 0.5, alternative="larger")
a1_, a2_ = tn[half1], tn[~half1]; Fv = a1_.var() / a2_.var(); Fp = 2 * min(sps.f.cdf(Fv, len(a1_) - 1, len(a2_) - 1), 1 - sps.f.cdf(Fv, len(a1_) - 1, len(a2_) - 1))
ct_ = pd.crosstab(np.where(tside < 0, "sell futures, buy ETF", "buy futures, sell ETF"), tn > 0); chi_ = sps.chi2_contingency(ct_)
CT = dict(n=len(tn), mean_bp=round(tn.mean(), 2), median_bp=round(tn.median(), 2), sd_bp=round(tn.std(), 2), skew=round(float(sps.skew(tn)), 2), kurt=round(float(sps.kurtosis(tn)), 2),
          t=round(float(t1.statistic), 2), p=round(float(t1.pvalue), 4), ci=[round(ci_[0], 2), round(ci_[1], 2)], wilcoxon_p=round(float(wil.pvalue), 4),
          wins=int((tn > 0).sum()), prop_z=round(float(pz[0]), 2), prop_p=round(float(pz[1]), 4),
          welch_p=round(float(sps.ttest_ind(a1_, a2_, equal_var=False).pvalue), 4), mw_p=round(float(sps.mannwhitneyu(a1_, a2_).pvalue), 4),
          F=round(float(Fv), 2), F_p=round(float(Fp), 4), chi2=round(float(chi_[0]), 3), chi_p=round(float(chi_[1]), 4), chi_table=ct_.values.tolist())
print(pd.Series(CT).to_string()); R["m15"]["tests"] = CT

n                            29
mean_bp                   23.68
median_bp                 18.04
sd_bp                      41.7
skew                       1.11
kurt                       1.93
t                          3.06
p                        0.0049
ci                [7.82, 39.54]
wilcoxon_p               0.0035
wins                         22
prop_z                     3.25
prop_p                   0.0006
welch_p                   0.663
mw_p                     0.7107
F                          1.59
F_p                      0.4074
chi2                      2.321
chi_p                    0.1277
chi_table     [[4, 4], [3, 18]]


### When do signals appear? One-way and two-way ANOVA (sessions 13–20)
The daily z-score grouped by week of the monthly expiry cycle (days to expiry), with Tukey HSD post hoc comparisons,
and a two-way ANOVA adding the expiry-day regime (Thursday until Aug-2025, Tuesday after).

In [25]:
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
A_ = pd.DataFrame({"z": d.z, "week": pd.cut(d.DTE, [-1, 6, 13, 20, 40], labels=["0-6 days", "7-13 days", "14-20 days", "21+ days"]),
                   "regime": np.where(d.date >= "2025-09-01", "Tuesday", "Thursday")}).dropna()
a1w = sm.stats.anova_lm(smf.ols("z ~ week", A_).fit(), typ=2); tk_ = pairwise_tukeyhsd(A_.z, A_.week.astype(str))
a2w = sm.stats.anova_lm(smf.ols("z ~ week * regime", A_).fit(), typ=2)
AN = dict(F=round(float(a1w.F.iloc[0]), 2), p=float(a1w["PR(>F)"].iloc[0]), means=A_.groupby("week", observed=True).z.mean().round(2).to_dict(),
          tukey=[f"{r_[0]} vs {r_[1]}" for r_ in tk_._results_table.data[1:] if r_[-1]], interaction_p=round(float(a2w["PR(>F)"].iloc[2]), 4),
          regime_p=round(float(a2w["PR(>F)"].iloc[1]), 4))
print(pd.Series(AN).to_string()); R["m15"]["anova"] = AN

F                                                            50.57
p                                                              0.0
means            {'0-6 days': -0.68, '7-13 days': -0.06, '14-20...
tukey            [0-6 days vs 14-20 days, 0-6 days vs 21+ days,...
interaction_p                                               0.4075
regime_p                                                    0.0867


### Testing regression assumptions on the Step 5 model (sessions 39–42)
Durbin–Watson (autocorrelation), Jarque–Bera (normality), Breusch–Pagan (heteroskedasticity) and Cook's distance (influential days).

In [26]:
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import het_breuschpagan
cook = pd.Series(eg5.get_influence().cooks_distance[0], index=d.date).sort_values(ascending=False)
DG = dict(dw=round(float(durbin_watson(eg5.resid)), 3), jb_p=float(jarque_bera(eg5.resid)[1]), bp_p=round(float(het_breuschpagan(eg5.resid, eg5.model.exog)[1]), 4),
          cook_top=[(str(k_.date()), round(float(v_), 4)) for k_, v_ in cook.head(3).items()], cook_cut=round(4 / len(d), 4), n_above=int((cook > 4 / len(d)).sum()))
print(pd.Series(DG).to_string()); R["m15"]["diag"] = DG

dw                                                      1.007
jb_p                                                      0.0
bp_p                                                     0.61
cook_top    [(2024-06-04, 0.089), (2026-03-30, 0.0423), (2...
cook_cut                                               0.0049
n_above                                                    25


### Signal generation as a classification problem (sessions 43–60)
Every 15-minute moment when futures become rich (z crosses above 2) is a candidate trade: buy the ETF, short the synthetic, hold to expiry.
Label = 1 if that trade beats the risk-free rate after costs. Logistic regression is compared with a decision tree, random forest and
gradient boosting on a **time split** (first 60% to train, last 40% to test).

In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
cF = COST["retail"]; cumf = np.r_[0, np.cumsum(RF * CAL / 365)]
vol20 = (d.dlnB.rolling(20).std() * np.sqrt(TD) * 100).shift(1).values; ex_last = np.where(ISEXP & ISLAST)[0]
rows = []
for k in np.where((Z > 2) & ~(np.r_[0, Z[:-1]] > 2))[0]:
    k0 = k + 1
    if k0 >= KB or ISEXP[k] or ISEXP[k0]: continue
    nxt = ex_last[ex_last > k0]
    if not len(nxt): continue
    k1 = nxt[0]; i0, i1 = DI[k0], DI[k1]
    price = -((LF[k1] - LF[k0]) - (LB[k1] - LB[k0]))
    cost = STTO[i0] * PREM[i0] + 2 * 2.5e-6 + cF["slip_F"] + 3e-5 * PREM[i0] + (cF["stamp_B"] + cF["oth_B"] + cF["slip_B"]) \
           + STTX[i1] * max(FP[k0] - FP[k1], 0) / FP[k1] + (cF["stt_B"] + cF["oth_B"] + cF["slip_B"])
    net = price - cost - (cumf[i1] - cumf[i0])
    rows.append(dict(y=int(net > 0), z=Z[k], dte=d.DTE.values[i0], hour=bars.dt.iloc[k].hour + bars.dt.iloc[k].minute / 60, vol20=vol20[i0]))
CL = pd.DataFrame(rows).dropna(); cut = int(len(CL) * 0.6); trn, tst = CL.iloc[:cut], CL.iloc[cut:]; FE = ["z", "dte", "hour", "vol20"]
lg_ = smf.logit("y ~ z + dte + hour + vol20", trn).fit(disp=0); pr_ = lg_.predict(tst)
LG = dict(n=len(CL), n_train=len(trn), n_test=len(tst), base_rate=round(float(tst.y.mean()), 3), coefs={k_: [round(float(v_), 3), round(float(lg_.pvalues[k_]), 4)] for k_, v_ in lg_.params.items()},
          auc_logit=round(float(roc_auc_score(tst.y, pr_)), 3), confusion=confusion_matrix(tst.y, (pr_ > 0.5).astype(int)).tolist())
for nm, mdl in [("tree", DecisionTreeClassifier(max_depth=3, random_state=0)), ("forest", RandomForestClassifier(300, max_depth=4, random_state=0)),
                ("boosting", GradientBoostingClassifier(max_depth=2, n_estimators=150, random_state=0))]:
    mdl.fit(trn[FE], trn.y); LG["auc_" + nm] = round(float(roc_auc_score(tst.y, mdl.predict_proba(tst[FE])[:, 1])), 3)
print(pd.Series(LG).to_string()); R["m15"]["classify"] = LG

n                                                             174
n_train                                                       104
n_test                                                         70
base_rate                                                   0.871
coefs           {'Intercept': [3.707, 0.3513], 'z': [0.853, 0....
auc_logit                                                   0.699
confusion                                      [[9, 0], [40, 21]]
auc_tree                                                    0.626
auc_forest                                                  0.774
auc_boosting                                                0.711


In [28]:
out_ = []
for who, trs in TR15.items():
    for t_ in trs:
        out_.append(dict(who=who, side=t_["side"], sig_dt=str(bars.dt.iloc[t_["ks"]]), sig_i=int(DI[t_["ks"]]), sig_lnF=LF[t_["ks"]], sig_lnB=LB[t_["ks"]], zsig=t_["zsig"],
                         entry_dt=str(bars.dt.iloc[t_["k0"]]), i0=t_["i0"], Fe=t_["Fe"], Be=t_["Be"], exit_dt=str(bars.dt.iloc[t_["k1"]]), i1=t_["i1"],
                         Fx=t_["Fx"], Bx=t_["Bx"], reason=t_["reason"], net=t_["net"]))
pd.DataFrame(out_).to_csv("m15_trades.csv", index=False)
R["charts"]["m15_eqn"] = list(np.round(np.cumsum(dly_s) * 100, 4)); R["charts"]["m15_eqg"] = list(np.round(np.cumsum(dly_g) * 100, 4))
R["charts"]["cook"] = list(np.round(eg5.get_influence().cooks_distance[0], 5))
json.dump(R, open("results.json", "w"), default=float, indent=1)
print("saved m15_trades.csv and results.json")

saved m15_trades.csv and results.json
